# AI 620 — Assignment 4 (PySpark)

**Google Colab:** run cells top to bottom. First cell installs PySpark and sets a working folder under `/content/DE_Assignment_4` (change `WORKDIR` there if you want). If you already have `data/` on Drive, mount Drive and set `WORKDIR` to that path instead.


10M clicks is slow on free Colab; the data cell uses a smaller default you can increase for the final run.

In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])

import os

try:
    import google.colab  # noqa: F401
    _colab = True
except ImportError:
    _colab = False

if _colab:
    WORKDIR = "/content/DE_Assignment_4"
    os.makedirs(WORKDIR, exist_ok=True)
    os.chdir(WORKDIR)
else:
    WORKDIR = os.getcwd()

print("cwd:", os.getcwd())

cwd: /content/DE_Assignment_4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import rand, col, current_timestamp, expr, struct, lit
from pyspark.sql.types import IntegerType

spark = SparkSession.builder.appName("DataGen").getOrCreate()

# Clickstream (10M events, nested JSON)
spark.range(1, 10000000) \
    .withColumn("event_id", col("id")) \
    .withColumn("user_id", (rand() * 10000).cast(IntegerType())) \
    .withColumn("event_ts", current_timestamp() - expr(f"INTERVAL {7} DAY") * rand() * 7) \
    .withColumn("page_data", struct(lit("home").alias("page"), (rand() * 100).alias("time_spent"))) \
    .select("event_id", "user_id", "event_ts", "page_data") \
    .write.mode("overwrite").json("data/clickstream_raw/")

# IoT data (500K records, CSV)
spark.range(1, 500000) \
    .selectExpr(
        "id as sensor_id",
        "cast(rand() * 10000 as int) as warehouse_id",
        "rand() * 50 as temp_celsius",
        "rand() * 80 as humidity_pct"
    ) \
    .coalesce(1).write.mode("overwrite").csv("data/iot_raw/", header=True)

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    IntegerType,
    TimestampType,
    StringType,
    DoubleType,
)

spark = (
    SparkSession.builder.appName("DataGen")
    .master("local[*]")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

Generate data if the folders are empty (same logic as the course data notebook). Lower `N_CLICK` if the session keeps timing out.

In [ ]:
from pathlib import Path
from pyspark.sql.types import IntegerType

N_CLICK = 800_000   # was 10M in original; bump if you have RAM/time
N_IOT = 500_000

p_click = Path("data/clickstream_raw")
p_iot = Path("data/iot_raw")

need_click = (not p_click.exists()) or (not any(p_click.iterdir()))
need_iot = (not p_iot.exists()) or (not any(p_iot.iterdir()))

if need_click:
    p_click.mkdir(parents=True, exist_ok=True)
    spark.range(1, N_CLICK + 1) \
        .withColumn("event_id", F.col("id")) \
        .withColumn("user_id", (F.rand() * 10000).cast(IntegerType())) \
        .withColumn("event_ts", F.current_timestamp() - F.expr("INTERVAL 7 DAY") * F.rand() * 7) \
        .withColumn("page_data", F.struct(F.lit("home").alias("page"), (F.rand() * 100).alias("time_spent"))) \
        .select("event_id", "user_id", "event_ts", "page_data") \
        .write.mode("overwrite").json(str(p_click))
    print("wrote clickstream json")
else:
    print("clickstream path already has files, skip gen")

if need_iot:
    p_iot.mkdir(parents=True, exist_ok=True)
    spark.range(1, N_IOT) \
        .selectExpr(
            "id as sensor_id",
            "cast(rand() * 10000 as int) as warehouse_id",
            "rand() * 50 as temp_celsius",
            "rand() * 80 as humidity_pct",
        ) \
        .coalesce(1).write.mode("overwrite").csv(str(p_iot), header=True)
    print("wrote iot csv")
else:
    print("iot path already has files, skip gen")

clickstream path already has files, skip gen
iot path already has files, skip gen


## Part 1

### 1.1 read json, no schema

In [ ]:
df_raw = spark.read.json("data/clickstream_raw/")

print("(a) inferred schema")
df_raw.printSchema()
print("sample")
df_raw.show(2, truncate=False)

(a) inferred schema
root
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- page_data: struct (nullable = true)
 |    |-- page: string (nullable = true)
 |    |-- time_spent: double (nullable = true)
 |-- user_id: long (nullable = true)

sample
+--------+------------------------+-------------------------+-------+
|event_id|event_ts                |page_data                |user_id|
+--------+------------------------+-------------------------+-------+
|5000000 |2026-04-13T02:26:11.524Z|{home, 23.73338812092375}|5184   |
|5000001 |2026-05-07T00:55:33.440Z|{home, 60.298720020537}  |9107   |
+--------+------------------------+-------------------------+-------+
only showing top 2 rows


(b) Inference only looks at a sample of the JSON so types can be wrong later (e.g. ids that start numeric then get letters). Also the inferred layout changes if upstream adds fields, which breaks anything downstream that assumed the old shape.

### 1.2 explicit schema

In [ ]:
click_schema = StructType(
    [
        StructField("event_id", LongType(), True),
        StructField("user_id", IntegerType(), True),
        StructField("event_ts", TimestampType(), True),
        StructField(
            "page_data",
            StructType(
                [
                    StructField("page", StringType(), True),
                    StructField("time_spent", DoubleType(), True),
                ]
            ),
            True,
        ),
    ]
)

df_click = spark.read.schema(click_schema).json("data/clickstream_raw/")
df_click.printSchema()
df_click.show(3, truncate=False)

root
 |-- event_id: long (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- page_data: struct (nullable = true)
 |    |-- page: string (nullable = true)
 |    |-- time_spent: double (nullable = true)

+--------+-------+-----------------------+-------------------------+
|event_id|user_id|event_ts               |page_data                |
+--------+-------+-----------------------+-------------------------+
|5000000 |5184   |2026-04-13 02:26:11.524|{home, 23.73338812092375}|
|5000001 |9107   |2026-05-07 00:55:33.44 |{home, 60.298720020537}  |
|5000002 |7386   |2026-03-30 04:17:53.672|{home, 18.81448509832815}|
+--------+-------+-----------------------+-------------------------+
only showing top 3 rows


### 1.3 iot csv permissive

In [ ]:
df_iot_raw = (
    spark.read.option("mode", "PERMISSIVE")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("data/iot_raw/", header=True)
)

# Spark 4 often drops _corrupt_record entirely when every line parses (no column to reference)
if "_corrupt_record" in df_iot_raw.columns:
    n_bad = df_iot_raw.filter(F.col("_corrupt_record").isNotNull()).count()
    print("(a) corrupt count", n_bad)
    if n_bad > 0:
        df_iot_raw.filter(F.col("_corrupt_record").isNotNull()).show(2, truncate=100)
    df_iot_clean = df_iot_raw.filter(F.col("_corrupt_record").isNull()).drop("_corrupt_record")
else:
    n_bad = 0
    print("(a) corrupt count", n_bad, "(no _corrupt_record column — Spark did not materialize it for all-clean CSV)")
    df_iot_clean = df_iot_raw

print("(b) clean count", df_iot_clean.count())

(a) corrupt count 0 (no _corrupt_record column — Spark did not materialize it for all-clean CSV)
(b) clean count 499999


## Part 2

### 2.1 clickstream

In [ ]:
df_cs = (
    df_click
    .withColumn("page", F.col("page_data.page"))
    .withColumn("time_spent", F.col("page_data.time_spent"))
    .drop("page_data")
    .filter(F.col("time_spent") <= 300)
    .withColumn("event_hour", F.date_trunc("hour", F.col("event_ts")))
)

df_cs.cache()
print("rows after dropping bots (time_spent > 300 filtered out)", df_cs.count())
df_cs.show(3, truncate=False)
print(df_cs.storageLevel)

rows after dropping bots (time_spent > 300 filtered out) 9999999
+--------+-------+-----------------------+----+-----------------+-------------------+
|event_id|user_id|event_ts               |page|time_spent       |event_hour         |
+--------+-------+-----------------------+----+-----------------+-------------------+
|5000000 |5184   |2026-04-13 02:26:11.524|home|23.73338812092375|2026-04-13 02:00:00|
|5000001 |9107   |2026-05-07 00:55:33.44 |home|60.298720020537  |2026-05-07 00:00:00|
|5000002 |7386   |2026-03-30 04:17:53.672|home|18.81448509832815|2026-03-30 04:00:00|
+--------+-------+-----------------------+----+-----------------+-------------------+
only showing top 3 rows
Disk Memory Deserialized 1x Replicated


(v) I cached `df_cs` because part 3 and part 4 both use it again. Otherwise Spark recomputes the whole chain from disk each time.

### 2.2 iot

CSV has no timestamp column. Part 3.2 wants 5 minute spacing so I added `reading_ts` by row_number inside each warehouse (fake clock). Real data would already have one.

In [ ]:
w_order = Window.partitionBy("warehouse_id").orderBy("sensor_id")

df_iot = (
    df_iot_clean
    .withColumn("sensor_id", F.col("sensor_id").cast("long"))
    .withColumn("warehouse_id", F.col("warehouse_id").cast("int"))
    .withColumn("temp_celsius", F.col("temp_celsius").cast("double"))
    .withColumn("rn", F.row_number().over(w_order))
    .withColumn(
        "reading_ts",
        F.from_unixtime(
            F.unix_timestamp(F.lit("2025-01-01 00:00:00")) + (F.col("rn") - 1) * 300
        ).cast("timestamp"),
    )
    .withColumn("temp_f", F.col("temp_celsius") * 9 / 5 + 32)
    .withColumn(
        "temp_status",
        F.when(F.col("temp_f") > 100, F.lit("CRITICAL"))
        .when((F.col("temp_f") > 85) & (F.col("temp_f") <= 100), F.lit("WARNING"))
        .otherwise(F.lit("NORMAL")),
    )
    .repartition(8)
)

print("partitions", df_iot.rdd.getNumPartitions())
df_iot.show(5, truncate=False)

partitions 8
+---------+------------+------------------+------------------+---+-------------------+------------------+-----------+
|sensor_id|warehouse_id|temp_celsius      |humidity_pct      |rn |reading_ts         |temp_f            |temp_status|
+---------+------------+------------------+------------------+---+-------------------+------------------+-----------+
|216581   |1956        |31.116596913892142|8.508138626658006 |28 |2025-01-01 02:15:00|88.00987444500586 |WARNING    |
|279348   |1436        |4.985923951920091 |9.596495438962167 |33 |2025-01-01 02:40:00|40.97466311345616 |NORMAL     |
|89350    |8532        |8.027310457971875 |50.74093529819735 |12 |2025-01-01 00:55:00|46.449158824349375|NORMAL     |
|220243   |9236        |29.34923577726748 |30.327941104968723|30 |2025-01-01 02:25:00|84.82862439908146 |NORMAL     |
|134115   |5758        |36.93476470780103 |41.744941660197775|18 |2025-01-01 01:25:00|98.48257647404185 |WARNING    |
+---------+------------+------------------+

(iv) `repartition` forces a shuffle and spreads rows across n parts. `coalesce` mostly merges blocks to fewer parts without a full shuffle when shrinking, so it is cheaper but you can get uneven files.

## Part 3

### 3.1 sessions (10 min gap)

In [ ]:
w_user = Window.partitionBy("user_id").orderBy("event_ts")

df_sess = (
    df_cs.withColumn("prev_event_ts", F.lag("event_ts").over(w_user))
    .withColumn(
        "gap_min",
        (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_event_ts")) / 60.0,
    )
    .withColumn(
        "session_start",
        F.when(F.col("prev_event_ts").isNull() | (F.col("gap_min") > 10), 1).otherwise(0),
    )
)

w_cum = Window.partitionBy("user_id").orderBy("event_ts").rowsBetween(Window.unboundedPreceding, 0)
df_sess = df_sess.withColumn("session_id", F.sum("session_start").over(w_cum))
df_sess.cache()

df_session_stats = df_sess.groupBy("user_id", "session_id").agg(
    ((F.unix_timestamp(F.max("event_ts")) - F.unix_timestamp(F.min("event_ts"))) / 60.0).alias(
        "session_duration_minutes"
    ),
    F.count("*").alias("total_clicks"),
)

df_session_stats.show(10, truncate=False)
print("session summary rows", df_session_stats.count())

df_cs.unpersist()

+-------+----------+------------------------+------------+
|user_id|session_id|session_duration_minutes|total_clicks|
+-------+----------+------------------------+------------+
|148    |1         |0.0                     |1           |
|148    |2         |0.0                     |1           |
|148    |3         |0.0                     |1           |
|148    |4         |0.0                     |1           |
|148    |5         |0.0                     |1           |
|148    |6         |0.0                     |1           |
|148    |7         |2.45                    |2           |
|148    |8         |0.0                     |1           |
|148    |9         |0.0                     |1           |
|148    |10        |6.75                    |2           |
+-------+----------+------------------------+------------+
only showing top 10 rows
session summary rows 8678389


DataFrame[event_id: bigint, user_id: int, event_ts: timestamp, page: string, time_spent: double, event_hour: timestamp]

### 3.2 rolling anomaly

`bucket` = unix time in 5 minute units. `rangeBetween(-12,0)` is then roughly one hour of history for the handout.

In [ ]:
df_iot_b = df_iot.withColumn("bucket", (F.unix_timestamp("reading_ts") / 300).cast("long"))

w_roll = Window.partitionBy("warehouse_id").orderBy("bucket").rangeBetween(-12, 0)

df_roll = df_iot_b.withColumn("roll_mean", F.avg("temp_f").over(w_roll)).withColumn(
    "roll_std", F.stddev("temp_f").over(w_roll)
)

df_anomaly = df_roll.withColumn(
    "is_anomaly",
    F.when(
        F.col("roll_std").isNull() | (F.col("roll_std") == 0) | F.isnan(F.col("roll_std")),
        False,
    ).otherwise(F.col("temp_f") > (F.col("roll_mean") + 3 * F.col("roll_std"))),
)

df_anomaly.filter("is_anomaly").show(10, truncate=False)
print("anomaly flags", df_anomaly.filter("is_anomaly").count())

+---------+------------+------------+------------+---+----------+------+-----------+------+---------+--------+----------+
|sensor_id|warehouse_id|temp_celsius|humidity_pct|rn |reading_ts|temp_f|temp_status|bucket|roll_mean|roll_std|is_anomaly|
+---------+------------+------------+------------+---+----------+------+-----------+------+---------+--------+----------+
+---------+------------+------------+------------+---+----------+------+-----------+------+---------+--------+----------+

anomaly flags 0


## Part 4

### 4.1 broadcast join

In [ ]:
lookup_rows = [(i, ["North", "South", "East", "West"][i % 4]) for i in range(100)]
df_lookup = spark.createDataFrame(lookup_rows, ["warehouse_id", "region"])

joined = df_anomaly.join(df_lookup.hint("broadcast"), "warehouse_id", "left")
joined.select("warehouse_id", "region", "temp_f").show(5, truncate=False)
joined.explain(mode="formatted")

+------------+------+-----------------+
|warehouse_id|region|temp_f           |
+------------+------+-----------------+
|5965        |NULL  |32.44365626319511|
|3554        |NULL  |46.96908447426154|
|9420        |NULL  |40.31504218924741|
|7737        |NULL  |44.02249950214514|
|5972        |NULL  |59.04553234777934|
+------------+------+-----------------+
only showing top 5 rows
== Physical Plan ==
AdaptiveSparkPlan (19)
+- Project (18)
   +- BroadcastHashJoin LeftOuter BuildRight (17)
      :- Project (13)
      :  +- Window (12)
      :     +- Sort (11)
      :        +- Exchange (10)
      :           +- Project (9)
      :              +- Exchange (8)
      :                 +- Project (7)
      :                    +- Project (6)
      :                       +- Window (5)
      :                          +- Sort (4)
      :                             +- Exchange (3)
      :                                +- Project (2)
      :                                   +- Scan csv  (1)

(b) Spark skips broadcast if the build side is bigger than the threshold (roughly single digit MB of serialized estimate). It also falls back if it cannot get sizes (stats off) or in some join types where broadcast is not allowed the way you wrote the query.

### 4.2 parquet + prune read

In [ ]:
out_hour = "output/clickstream_by_hour"
df_cs.write.mode("overwrite").partitionBy("event_hour").parquet(out_hour)

h0 = df_cs.select(F.min("event_hour").alias("h")).first()["h"]
print("filter hour", h0)

df_pruned = spark.read.parquet(out_hour).filter(F.col("event_hour") == F.lit(h0))
df_pruned.show(3, truncate=False)
# Spark 4: use extended (old explain(true)); "true" is not a valid mode anymore
df_pruned.explain(mode="extended")

filter hour 2026-03-23 04:00:00
+--------+-------+-----------------------+----+------------------+-------------------+
|event_id|user_id|event_ts               |page|time_spent        |event_hour         |
+--------+-------+-----------------------+----+------------------+-------------------+
|1018752 |5776   |2026-03-23 04:35:33.63 |home|72.67245086896683 |2026-03-23 04:00:00|
|1019651 |9962   |2026-03-23 04:32:30.658|home|7.260657526831871 |2026-03-23 04:00:00|
|1019698 |271    |2026-03-23 04:54:17.487|home|14.072661178790124|2026-03-23 04:00:00|
+--------+-------+-----------------------+----+------------------+-------------------+
only showing top 3 rows
== Parsed Logical Plan ==
'Filter '`=`('event_hour, 2026-03-23 04:00:00)
+- Relation [event_id#3982L,user_id#3983,event_ts#3984,page#3985,time_spent#3986,event_hour#3987] parquet

== Analyzed Logical Plan ==
event_id: bigint, user_id: int, event_ts: timestamp, page: string, time_spent: double, event_hour: timestamp
Filter (event_hour

## Part 5

### 5.1 append sessions by date

`event_date` = date part of `event_ts` on each row of the sessionized click table.

In [ ]:
df_sessionized = df_sess.withColumn("event_date", F.to_date("event_ts"))

out_sess = "output/sessions_by_date"
# rm -rf this folder in Colab file browser if you rerun and do not want duplicate appends
df_sessionized.write.mode("append").partitionBy("event_date").parquet(out_sess)

df_sessionized.select("event_id", "user_id", "event_ts", "session_id", "event_date").show(5, truncate=False)
df_sess.unpersist()

+--------+-------+-----------------------+----------+----------+
|event_id|user_id|event_ts               |session_id|event_date|
+--------+-------+-----------------------+----------+----------+
|2749254 |148    |2026-03-23 04:27:34.752|1         |2026-03-23|
|289510  |148    |2026-03-23 06:42:59.948|2         |2026-03-23|
|4762525 |148    |2026-03-23 07:07:27.178|3         |2026-03-23|
|2091331 |148    |2026-03-23 07:33:10.647|4         |2026-03-23|
|3438116 |148    |2026-03-23 08:41:41.826|5         |2026-03-23|
+--------+-------+-----------------------+----------+----------+
only showing top 5 rows


DataFrame[event_id: bigint, user_id: int, event_ts: timestamp, page: string, time_spent: double, event_hour: timestamp, prev_event_ts: timestamp, gap_min: double, session_start: int, session_id: bigint]

(c) Append is only safe if each run writes new data once (e.g. daily batch). Rerun the whole notebook on the same clicks and you duplicate partitions. For real idempotence you would overwrite that date partition or merge on keys.

### 5.2 one csv file

In [ ]:
out_csv = "output/anomalies_single"
df_anomaly.coalesce(1).write.mode("overwrite").option("header", True).csv(out_csv)
print("csv folder", out_csv)

csv folder output/anomalies_single


(b) `coalesce(1)` pushes everything through one task so you lose parallel write and risk OOM on big data.

(c) For huge data write many parts or columnar formats (parquet) to cloud storage, then compact offline if you really need one file.

## Bonus AQE

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
print(spark.conf.get("spark.sql.adaptive.enabled"))

true


AQE looks at runtime sizes after shuffles and can change join type, merge tiny partitions, etc. I also turned it on in the builder above.

## Short answers for the pdf questions

New nested field under `page_data`: with a fixed `StructType` Spark will not show it unless you add it to the schema (it gets dropped on read). New top level field same story unless you extend the schema.

Same session ids on rerun: only if the same rows show up in the same order with the same timestamps. Change ordering or data and the running sum of session_start flags changes.

2 cores: big shuffles or `coalesce(1)` are the usual pain (slow or OOM). Fix by more executors/cores, smaller shuffle (`spark.sql.shuffle.partitions`), avoid single file coalesce, broadcast small sides when possible.

In [ ]:
spark.stop()